# 1.7 LLM 数据增强 (LLM Data Augmentation)

> 🕐 预估学习时间：30分钟

数据增强是缓解低资源场景、提升模型鲁棒性与多样性的核心技术。在 LLM 时代，从传统文本增强到指令演化方法，数据增强的工程实现发生了深刻变化。

本节涵盖：
- 数据增强概述与质量评估框架
- 回译与释义（Back-Translation / Paraphrasing）
- Self-Instruct 与 Evol-Instruct 自动指令生成
- 对话数据增强（多轮、角色反转、上下文扰动）
- 数据质量控制（去重、多样性度量、过滤）

## 1. 数据增强概述

**为什么 LLM 需要数据增强**：
- **低资源场景**：垂直领域指令数据稀缺，增强可扩充训练集
- **多样性提升**：避免模型过拟合特定表达，提升泛化能力
- **鲁棒性增强**：让模型面对 paraphrase、噪声等扰动时保持稳定
- **对齐数据扩展**：RLHF/DPO 需要大量偏好对，增强可降低标注成本

**传统 NLP 增强 vs LLM 专用增强**：
| 类别 | 传统方法 | LLM 时代方法 |
|------|---------|-------------|
| 词级 | 同义词替换、随机删除 | LLM 释义、回译 |
| 句级 | 句序打乱、模板填充 | Self-Instruct、Evol-Instruct |
| 对话级 | 角色互换、上下文裁剪 | 多轮演化、约束注入 |

**增强质量三要素**：
- **保真度 (Fidelity)**：增强后语义是否与原样本一致
- **多样性 (Diversity)**：增强样本之间是否足够不同
- **覆盖率 (Coverage)**：是否覆盖了目标分布的多种表达

三者相互制约：保真度过高则多样性不足，多样性过高则保真度下降。

In [ ]:
import torch
import numpy as np
import random
from collections import Counter, defaultdict

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)


class DataSample:
    # 训练样本表示：指令 + 输入 + 输出 + 元信息

    def __init__(self, instruction, output, input_text='', meta=None):
        self.instruction = instruction
        self.input_text = input_text
        self.output = output
        self.meta = meta or {}

    def text(self):
        if self.input_text:
            return self.instruction + ' ' + self.input_text + ' ' + self.output
        return self.instruction + ' ' + self.output

    def __repr__(self):
        return f'DataSample(instr={self.instruction[:30]}..., out={self.output[:30]}...)'


class AugmentationEvaluator:
    # 评估增强质量：多样性、保真度、覆盖率

    def __init__(self):
        self.vocab = set()

    def _tokenize(self, text):
        return text.lower().split()

    def diversity(self, samples):
        # 用 unique bigram 比例衡量多样性
        all_ngrams = []
        for s in samples:
            toks = self._tokenize(s)
            ngrams = ['_'.join(toks[i:i+2]) for i in range(len(toks)-1)]
            all_ngrams.extend(ngrams)
        if not all_ngrams:
            return 0.0
        return len(set(all_ngrams)) / len(all_ngrams)

    def fidelity(self, original, augmented):
        # 用 Jaccard 相似度近似语义保真度
        a = set(self._tokenize(original))
        b = set(self._tokenize(augmented))
        if not a and not b:
            return 1.0
        return len(a & b) / max(len(a | b), 1)

    def coverage(self, samples, reference_vocab):
        # 增强样本对参考词表的覆盖率
        vocab = set()
        for s in samples:
            vocab.update(self._tokenize(s))
        return len(vocab & reference_vocab) / max(len(reference_vocab), 1)

    def evaluate(self, original, augmented_list, reference_vocab=None):
        div = self.diversity(augmented_list)
        fid = [self.fidelity(original, aug) for aug in augmented_list]
        avg_fid = sum(fid) / len(fid) if fid else 0.0
        cov = self.coverage(augmented_list, reference_vocab or set())
        return {
            'diversity': round(div, 3),
            'fidelity': round(avg_fid, 3),
            'coverage': round(cov, 3),
        }


print('=== 数据增强框架 ===')

evaluator = AugmentationEvaluator()
original = 'translate this sentence to chinese please'
augmented = [
    'please render this sentence into chinese',
    'convert this sentence to the chinese language',
    'kindly translate the following sentence into chinese',
]
ref_vocab = set('translate sentence chinese please convert render language'.split())

metrics = evaluator.evaluate(original, augmented, ref_vocab)
print(f'\n原始样本: {original}')
print(f'增强样本数: {len(augmented)}')
for k, v in metrics.items():
    print(f'  {k}: {v}')

sample = DataSample('Summarize the article', 'The article discusses AI.')
print(f'\n样本表示: {sample}')
print(f'拼接文本: {sample.text()}')

print(f'\nKey: 数据增强需同时兼顾保真度、多样性与覆盖率，三者构成质量三角。')

## 2. 回译与释义

**回译 (Back-Translation)**：
- 将文本 A 翻译到中间语言 B，再翻译回 A，得到语义相近但表达不同的变体
- 优点：自然引入词汇与句式变化，适合指令数据扩充
- 缺点：依赖翻译模型质量，可能引入语义漂移

**释义 (Paraphrasing)**：
- 通过模板、同义词替换、句式改写生成同义表达
- 模板法：定义句式骨架，填充不同槽位
- 同义词法：基于词典替换非关键词
- LLM 法：直接用大模型生成多种释义

**关键工程考量**：
- 保留指令的核心意图（任务动词如 translate、summarize 不宜替换）
- 控制改写幅度，避免语义偏移
- 对结构化输出（代码、JSON）慎用增强

In [ ]:
import torch
import random
from collections import defaultdict

torch.manual_seed(42)
random.seed(42)


class BackTranslator:
    # 模拟回译：通过预定义双语词典实现 en<->zh 的伪回译

    def __init__(self):
        # 简化的英中词典（演示用）
        self.en2zh = {
            'translate': '翻译', 'sentence': '句子', 'please': '请',
            'this': '这个', 'to': '到', 'chinese': '中文', 'english': '英文',
            'summarize': '总结', 'article': '文章', 'the': '这个',
            'write': '写', 'poem': '诗', 'about': '关于', 'nature': '自然',
        }
        self.zh2en = {v: k for k, v in self.en2zh.items()}
        # 同义词扩展，让回译产生变化
        self.synonyms = {
            'translate': ['convert', 'render'],
            'sentence': ['phrase', 'statement'],
            'please': ['kindly'],
            'summarize': ['condense', 'recap'],
            'article': ['passage', 'text'],
            'write': ['compose', 'draft'],
            'poem': ['verse', 'poetry'],
        }

    def _en_to_zh(self, text):
        words = text.lower().split()
        zh_words = [self.en2zh.get(w, w) for w in words]
        return ''.join(zh_words)

    def _zh_to_en(self, zh_text):
        # 贪心匹配最长中文词
        result = []
        i = 0
        while i < len(zh_text):
            matched = False
            for length in (3, 2, 1):
                chunk = zh_text[i:i+length]
                if chunk in self.zh2en:
                    base = self.zh2en[chunk]
                    candidates = [base]
                    if base in self.synonyms:
                        candidates.extend(self.synonyms[base])
                    result.append(random.choice(candidates))
                    i += length
                    matched = True
                    break
            if not matched:
                i += 1
        return ' '.join(result)

    def augment(self, text, n=1):
        results = []
        for _ in range(n):
            zh = self._en_to_zh(text)
            back = self._zh_to_en(zh)
            if back != text:
                results.append(back)
        return results


class Paraphraser:
    # 模板化释义：基于句式骨架与同义词替换生成 paraphrase

    TEMPLATES = [
        '{action} the following {obj} into {target}',
        'please {action} this {obj} to {target}',
        'kindly {action} the {obj} below to {target}',
        'i need you to {action} this {obj} to {target}',
    ]

    SYNONYMS = {
        'translate': ['convert', 'render', 'transform'],
        'summarize': ['condense', 'recap', 'digest'],
        'write': ['compose', 'draft', 'create'],
        'sentence': ['phrase', 'statement', 'text'],
        'article': ['passage', 'document', 'text'],
        'poem': ['verse', 'poetry', 'stanza'],
        'chinese': ['mandarin', 'cn'],
        'english': ['en'],
    }

    def _fill(self, template, slots):
        action = random.choice(self.SYNONYMS.get(slots['action'], [slots['action']]))
        obj = random.choice(self.SYNONYMS.get(slots['obj'], [slots['obj']]))
        target = random.choice(self.SYNONYMS.get(slots['target'], [slots['target']]))
        return template.format(action=action, obj=obj, target=target)

    def augment(self, slots, n=3):
        results = []
        for _ in range(n):
            tpl = random.choice(self.TEMPLATES)
            results.append(self._fill(tpl, slots))
        return list(dict.fromkeys(results))  # 去重保序


print('=== 回译与释义 ===')

bt = BackTranslator()
text = 'translate this sentence to chinese'
back_results = bt.augment(text, n=3)
print(f'\n原始: {text}')
print(f'回译结果:')
for i, r in enumerate(back_results, 1):
    print(f'  {i}. {r}')

pp = Paraphraser()
slots = {'action': 'translate', 'obj': 'sentence', 'target': 'chinese'}
para_results = pp.augment(slots, n=4)
print(f'\n释义结果:')
for i, r in enumerate(para_results, 1):
    print(f'  {i}. {r}')

# 演示指令数据增强
instruction_samples = [
    ('translate this sentence to chinese', {'action': 'translate', 'obj': 'sentence', 'target': 'chinese'}),
    ('summarize the article below', {'action': 'summarize', 'obj': 'article', 'target': 'brief'}),
    ('write a poem about nature', {'action': 'write', 'obj': 'poem', 'target': 'nature'}),
]
print(f'\n批量指令增强:')
for inst, s in instruction_samples:
    bt_aug = bt.augment(inst, n=1)
    pp_aug = pp.augment(s, n=1)
    combined = (bt_aug + pp_aug)[0] if (bt_aug + pp_aug) else inst
    print(f'  原: {inst}')
    print(f'  增: {combined}')

print(f'\nKey: 回译通过中间语言产生自然变体，模板释义可控且低成本，二者互补使用。')

## 3. Self-Instruct 与 Evol-Instruct

**Self-Instruct**：
- 用少量种子指令（通常 100~200 条）提示 LLM 自动生成新指令
- 流程：种子 → 生成 → 过滤 → 去重 → 入库
- 论文 *Self-Instruct (2022)* 用 175 条种子生成 52k 指令

**Evol-Instruct**：
- 在已有指令基础上进行演化，逐步增加复杂度或约束
- 演化方向：
  - **深化 (Deepen)**：增加推理步骤、引入边界条件
  - **简化 (Simplify)**：降低难度，覆盖基础场景
  - **约束 (Constrain)**：加入格式、长度、风格限制
  - **改写 (Rewrite)**：变换表达但保持意图

**工程要点**：
- 生成后必须做去重与质量过滤（否则会大量重复）
- 用 ROUGE-L 或 n-gram 重叠度过滤相似指令
- 控制演化深度，避免指令变得不可解

In [ ]:
import torch
import random
from collections import defaultdict

torch.manual_seed(42)
random.seed(42)


class SelfInstructGenerator:
    # 从种子指令出发，通过模板组合生成新指令

    VERBS = ['write', 'translate', 'summarize', 'explain', 'compare',
             'classify', 'rewrite', 'analyze']
    OBJECTS = ['a poem', 'an article', 'a sentence', 'a paragraph',
               'a review', 'a story', 'a report']
    TOPICS = ['nature', 'technology', 'history', 'science',
              'culture', 'sports', 'business', 'health']
    CONSTRAINTS = ['in formal tone', 'within 100 words', 'with examples',
                   'step by step', 'in chinese', 'concisely']

    def __init__(self, seed_instructions):
        self.seeds = seed_instructions
        self.generated = []

    def _generate_one(self):
        verb = random.choice(self.VERBS)
        obj = random.choice(self.OBJECTS)
        topic = random.choice(self.TOPICS)
        constraint = random.choice(self.CONSTRAINTS)
        templates = [
            f'{verb} {obj} about {topic} {constraint}',
            f'please {verb} {obj} on the topic of {topic} {constraint}',
            f'{verb} {obj} related to {topic}, {constraint}',
        ]
        return random.choice(templates)

    def generate(self, n=20):
        results = []
        for _ in range(n):
            results.append(self._generate_one())
        # 简单去重
        results = list(dict.fromkeys(results))
        self.generated = results
        return results

    def filter_by_overlap(self, threshold=0.5):
        # 用 token Jaccard 过滤与种子过于相似的指令
        def toks(s):
            return set(s.lower().split())

        filtered = []
        for instr in self.generated:
            tok_set = toks(instr)
            too_similar = False
            for seed in self.seeds:
                seed_set = toks(seed)
                if seed_set and tok_set:
                    jaccard = len(tok_set & seed_set) / len(tok_set | seed_set)
                    if jaccard > threshold:
                        too_similar = True
                        break
            if not too_similar:
                filtered.append(instr)
        return filtered


class EvolInstructGenerator:
    # 对已有指令进行演化：深化、简化、加约束、改写

    DEEPEN_PREFIXES = [
        'consider edge cases and ',
        'with detailed reasoning, ',
        'addressing potential counterarguments, ',
    ]
    CONSTRAINTS = [
        'within 50 words',
        'in bullet points',
        'with at least two examples',
        'in a neutral tone',
        'avoiding jargon',
    ]
    REWRITE_PREFIXES = [
        'in your own words, ',
        'rephrased clearly, ',
        'as if explaining to a beginner, ',
    ]

    def evolve(self, instruction, mode='deepen'):
        if mode == 'deepen':
            prefix = random.choice(self.DEEPEN_PREFIXES)
            return prefix + instruction
        elif mode == 'simplify':
            # 移除修饰词，截断到第一个逗号
            simplified = instruction.replace('detailed ', '').replace('comprehensive ', '')
            return simplified.split(',')[0]
        elif mode == 'constrain':
            c = random.choice(self.CONSTRAINTS)
            return instruction + ', ' + c
        elif mode == 'rewrite':
            prefix = random.choice(self.REWRITE_PREFIXES)
            return prefix + instruction
        return instruction

    def evolve_batch(self, instructions, modes=None):
        modes = modes or ['deepen', 'simplify', 'constrain', 'rewrite']
        results = []
        for instr in instructions:
            mode = random.choice(modes)
            results.append((mode, self.evolve(instr, mode)))
        return results


print('=== Self-Instruct 与 Evol-Instruct ===')

seeds = [
    'write a poem about nature',
    'translate this sentence to chinese',
    'summarize the article below',
    'explain how neural networks work',
]

sig = SelfInstructGenerator(seeds)
raw = sig.generate(n=30)
filtered = sig.filter_by_overlap(threshold=0.4)
print(f'\n种子指令数: {len(seeds)}')
print(f'生成指令数 (去重后): {len(raw)}')
print(f'过滤后指令数: {len(filtered)}')
print(f'\n样本指令:')
for i, instr in enumerate(filtered[:5], 1):
    print(f'  {i}. {instr}')

evolver = EvolInstructGenerator()
print(f'\n指令演化演示:')
base = 'explain how neural networks work'
for mode in ['deepen', 'simplify', 'constrain', 'rewrite']:
    evolved = evolver.evolve(base, mode)
    print(f'  {mode:10s}: {evolved}')

print(f'\nKey: Self-Instruct 从种子批量生成指令，Evol-Instruct 通过演化方向扩展指令空间。')

## 4. 对话数据增强

**多轮对话增强的挑战**：
- 上下文一致性：增强不能破坏对话连贯性
- 角色一致性：用户/助手风格需保持稳定
- 任务完整性：增强后仍需达成对话目标

**常用增强策略**：
- **回复变体 (Response Variation)**：为同一上下文生成多个等价回复
- **上下文裁剪 (Context Trimming)**：删除部分历史轮次，训练模型对长上下文的鲁棒性
- **角色扩展 (Role Expansion)**：在对话中插入系统提示或工具调用轮
- **意图扰动 (Intent Perturbation)**：改写用户提问方式但保持意图

**注意事项**：
- 不要破坏对话中的指代关系（如 it、上文提到）
- 工具调用对话需保持调用-返回配对
- 多轮对话增强后需重新检查对话状态

In [ ]:
import torch
import random
from collections import defaultdict

torch.manual_seed(42)
random.seed(42)


class DialogueAugmentor:
    # 多轮对话增强：回复变体、上下文裁剪、角色扩展、意图扰动

    RESPONSE_VARIANTS = {
        'greeting': [
            'hello! how can i help you today?',
            'hi there! what can i do for you?',
            'hey! how may i assist you?',
        ],
        'help_offer': [
            'sure, i can help with that.',
            'of course! let me take care of it.',
            'no problem, i will handle that for you.',
        ],
        'closing': [
            'is there anything else i can help with?',
            'let me know if you need further assistance.',
            'anything else i can do for you?',
        ],
    }

    INTENT_REWRITES = {
        'i want to buy a phone': [
            'i am looking to purchase a phone',
            'i need a new phone',
            'help me find a phone to buy',
        ],
        'what is the price': [
            'how much does it cost',
            'could you tell me the price',
            'what does it run for',
        ],
    }

    def response_variation(self, dialogue, intent_key):
        # 替换助手回复为同义变体
        variants = self.RESPONSE_VARIANTS.get(intent_key, [])
        if not variants:
            return [dict(t) for t in dialogue]
        new_dialogue = []
        for turn in dialogue:
            if turn['role'] == 'assistant' and turn['content'] in variants:
                new_choice = random.choice([v for v in variants if v != turn['content']] or variants)
                new_dialogue.append({'role': 'assistant', 'content': new_choice})
            else:
                new_dialogue.append(dict(turn))
        return new_dialogue

    def context_trimming(self, dialogue, keep_recent=2):
        # 只保留系统提示 + 最近 N 轮
        result = [t for t in dialogue if t['role'] == 'system']
        non_system = [t for t in dialogue if t['role'] != 'system']
        result.extend(non_system[-keep_recent:])
        return result

    def role_expansion(self, dialogue, system_extra=None):
        # 插入额外系统提示
        result = [dict(t) for t in dialogue]
        if system_extra:
            insert_idx = 1 if result and result[0]['role'] == 'system' else 0
            result.insert(insert_idx, {'role': 'system', 'content': system_extra})
        return result

    def intent_perturbation(self, dialogue):
        # 改写用户提问但保持意图
        result = []
        for turn in dialogue:
            if turn['role'] == 'user':
                content = turn['content'].lower().strip()
                if content in self.INTENT_REWRITES:
                    new_content = random.choice(self.INTENT_REWRITES[content])
                    result.append({'role': 'user', 'content': new_content})
                    continue
            result.append(dict(turn))
        return result

    def augment_all(self, dialogue, intent_key='help_offer'):
        return {
            'response_variation': self.response_variation(dialogue, intent_key),
            'context_trimming': self.context_trimming(dialogue, keep_recent=2),
            'role_expansion': self.role_expansion(dialogue, 'be concise and friendly'),
            'intent_perturbation': self.intent_perturbation(dialogue),
        }


def print_dialogue(label, dialogue):
    print(f'\n--- {label} ({len(dialogue)} 轮) ---')
    for t in dialogue:
        role = t['role']
        content = t['content']
        print(f'  [{role}] {content}')


print('=== 对话数据增强 ===')

# 客服对话示例
dialogue = [
    {'role': 'system', 'content': 'you are a helpful shopping assistant'},
    {'role': 'user', 'content': 'i want to buy a phone'},
    {'role': 'assistant', 'content': 'sure, i can help with that.'},
    {'role': 'user', 'content': 'what is the price'},
    {'role': 'assistant', 'content': 'is there anything else i can help with?'},
]

augmentor = DialogueAugmentor()
augmented = augmentor.augment_all(dialogue, intent_key='help_offer')

print_dialogue(f'原始对话', dialogue)

for method, aug_dialogue in augmented.items():
    print_dialogue(method, aug_dialogue)

print(f'\nKey: 对话增强需保证上下文一致与角色稳定，裁剪与变体是两种最常用策略。')

## 5. 数据质量控制

**为什么需要质量控制**：
- 增强数据天然带有噪声，低质量样本会拖累训练
- 重复样本导致训练梯度偏移，浪费算力
- 多样性不足的增强等于没增强

**核心控制环节**：
- **去重 (Deduplication)**：基于 n-gram 或 embedding 相似度移除近似重复
- **多样性度量 (Diversity)**：用 unique n-gram 比例、自信息熵衡量
- **质量过滤 (Quality Filter)**：基于长度、格式、关键词、模型评分过滤
- **分布检查 (Distribution Check)**：确保增强后分布与目标一致

**工程实践**：
- 增强后通常保留原数据，按比例混合（如 1:3）
- 用 MinHash / SimHash 做大规模去重
- 用 reward model 或 LLM-as-judge 给增强样本打分

In [ ]:
import torch
import numpy as np
import random
from collections import Counter, defaultdict

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)


class AugmentationQualityControl:
    # 增强数据质量控制：去重、多样性、质量过滤

    def __init__(self, min_length=5, max_length=200, dedup_threshold=0.8):
        self.min_length = min_length
        self.max_length = max_length
        self.dedup_threshold = dedup_threshold

    def _tokenize(self, text):
        return text.lower().split()

    def length_filter(self, samples):
        kept = []
        for s in samples:
            n = len(self._tokenize(s))
            if self.min_length <= n <= self.max_length:
                kept.append(s)
        return kept

    def jaccard(self, a, b):
        sa = set(self._tokenize(a))
        sb = set(self._tokenize(b))
        if not sa and not sb:
            return 1.0
        return len(sa & sb) / max(len(sa | sb), 1)

    def deduplicate(self, samples):
        # 贪心去重：与已保留样本相似度超阈值则丢弃
        kept = []
        for s in samples:
            is_dup = False
            for k in kept:
                if self.jaccard(s, k) >= self.dedup_threshold:
                    is_dup = True
                    break
            if not is_dup:
                kept.append(s)
        return kept

    def diversity_score(self, samples):
        # 用 unique bigram 比例 + 词熵衡量多样性
        bigrams = []
        tokens = []
        for s in samples:
            t = self._tokenize(s)
            tokens.extend(t)
            bigrams.extend(['_'.join(t[i:i+2]) for i in range(len(t)-1)])
        unique_bigram_ratio = len(set(bigrams)) / max(len(bigrams), 1)
        # 词熵
        freq = Counter(tokens)
        total = sum(freq.values())
        probs = [c / total for c in freq.values() if c > 0]
        entropy = -sum(p * np.log2(p) for p in probs) if probs else 0.0
        return {
            'unique_bigram_ratio': round(unique_bigram_ratio, 3),
            'token_entropy': round(entropy, 3),
            'vocab_size': len(freq),
        }

    def quality_score(self, sample):
        # 简单质量评分：长度归一 + 词汇丰富度
        toks = self._tokenize(sample)
        if not toks:
            return 0.0
        length_score = min(len(toks) / 20, 1.0)
        richness = len(set(toks)) / len(toks)
        return round(0.5 * length_score + 0.5 * richness, 3)

    def run_pipeline(self, samples):
        n0 = len(samples)
        after_len = self.length_filter(samples)
        n1 = len(after_len)
        after_dedup = self.deduplicate(after_len)
        n2 = len(after_dedup)
        div = self.diversity_score(after_dedup)
        scores = [self.quality_score(s) for s in after_dedup]
        avg_score = round(sum(scores) / len(scores), 3) if scores else 0.0
        return {
            'original_count': n0,
            'after_length_filter': n1,
            'after_dedup': n2,
            'diversity': div,
            'avg_quality_score': avg_score,
            'clean_samples': after_dedup,
        }


print('=== 数据质量控制 ===')

# 模拟一批增强指令（含重复与低质量样本）
raw_samples = [
    'translate this sentence to chinese',
    'translate this sentence to chinese',  # 完全重复
    'convert this sentence to the chinese language',
    'hi',  # 过短
    'please render this sentence into chinese',
    'translate this phrase to chinese please',
    'translate this sentence to chinese',  # 重复
    'summarize the article below concisely',
    'summarize the article below in brief',
    'write a poem about nature with examples',
]

qc = AugmentationQualityControl(min_length=4, dedup_threshold=0.7)
report = qc.run_pipeline(raw_samples)

print(f'\n原始样本数: {report["original_count"]}')
print(f'长度过滤后: {report["after_length_filter"]}')
print(f'去重后: {report["after_dedup"]}')
print(f'\n多样性指标:')
for k, v in report['diversity'].items():
    print(f'  {k}: {v}')
print(f'\n平均质量分: {report["avg_quality_score"]}')
print(f'\n清洗后样本:')
for i, s in enumerate(report['clean_samples'], 1):
    score = qc.quality_score(s)
    print(f'  {i}. [{score}] {s}')

print(f'\nKey: 质量控制是增强的最后一道闸门，去重 + 多样性度量 + 质量评分缺一不可。')

## 6. Magpie 数据生成方法

Magpie 是一种利用 LLM 对齐特性的数据生成方法。核心发现：经过对齐训练的 LLM（如 Llama-2-Chat），只需要给定对话模板的开头，就能自动生成高质量的指令-回复对。

### 工作原理
1. 构造对话模板前缀（如 `<s>[INST]`）
2. 让 LLM 续写，它会自动生成一个指令
3. 再让 LLM 回答这个指令
4. 无需种子指令，无需人工干预

### 优势
- 完全自动化，无需种子数据
- 生成的指令多样性高
- 数据质量接近人工标注
- 可生成任意规模的数据集

### 与 Self-Instruct 的区别
- Self-Instruct 需要种子指令来引导生成
- Magpie 利用对齐特性，无需种子
- Magpie 生成的指令分布更接近真实用户输入

In [ ]:
import torch
import numpy as np
import random
from collections import Counter

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)


class MagpieGenerator:
    # 模拟 Magpie 数据生成：利用对齐 LLM 的对话模板前缀自动产生指令-回复对

    # 对齐 LLM 的对话模板前缀（模拟 Llama-2-Chat 的 [INST] 标记）
    DIALOG_PREFIX = '<s>[INST]'

    # 模拟对齐 LLM 从前缀续写出的指令模式（无需种子）
    INSTRUCTION_TEMPLATES = [
        'explain the difference between {a} and {b}',
        'write a {style} about {topic}',
        'how does {concept} work in {field}',
        'what are the main benefits of {topic}',
        'compare {a} and {b} in terms of {aspect}',
        'summarize the key points of {topic}',
        'list {n} practical examples of {topic}',
        'describe the process of {process} step by step',
        'analyze the impact of {topic} on {field}',
        'why is {topic} important for {field}',
    ]

    TOPICS = ['machine learning', 'climate change', 'quantum computing',
              'renewable energy', 'neural networks', 'data privacy',
              'blockchain', 'genetic engineering', 'space exploration',
              'cybersecurity']

    STYLES = ['poem', 'essay', 'tutorial', 'review', 'story']
    ASPECTS = ['cost', 'performance', 'complexity', 'scalability']

    def _sample_slots(self):
        return {
            'a': random.choice(self.TOPICS),
            'b': random.choice(self.TOPICS),
            'style': random.choice(self.STYLES),
            'topic': random.choice(self.TOPICS),
            'concept': random.choice(self.TOPICS),
            'field': random.choice(self.TOPICS),
            'aspect': random.choice(self.ASPECTS),
            'n': str(random.randint(3, 7)),
            'process': random.choice(self.TOPICS),
        }

    def generate_instruction(self):
        # 模拟对齐 LLM 从对话前缀自动续写出指令
        template = random.choice(self.INSTRUCTION_TEMPLATES)
        slots = self._sample_slots()
        # 避免 a == b
        while slots['a'] == slots['b']:
            slots['b'] = random.choice(self.TOPICS)
        instruction = template.format(**slots)
        full_template = self.DIALOG_PREFIX + ' ' + instruction + ' [/INST]'
        return {
            'prefix': self.DIALOG_PREFIX,
            'instruction': instruction,
            'full_template': full_template,
            'slots': slots,
        }

    def generate_response(self, instruction_info):
        # 模拟对齐 LLM 对指令的回复
        slots = instruction_info['slots']
        instruction = instruction_info['instruction']
        first_word = instruction.split()[0].lower()
        responses = {
            'explain': 'Here is the explanation: {a} focuses on..., while {b} emphasizes...',
            'write': 'Here is a {style} about {topic}: (content)...',
            'how': '{concept} works in {field} through the following mechanism: ...',
            'what': 'The main benefits of {topic} include: 1)... 2)... 3)...',
            'compare': 'Comparison of {a} and {b} on {aspect}: ...',
            'summarize': 'Key points of {topic}: - point 1 - point 2 - point 3',
            'list': 'Here are {n} examples of {topic}: 1)... 2)... 3)...',
            'describe': 'The process of {process}: Step 1... Step 2... Step 3...',
            'analyze': 'Impact analysis of {topic} on {field}: ...',
            'why': '{topic} is important for {field} because: 1)... 2)...',
        }
        template = responses.get(first_word, 'Here is a response to the query about {topic}.')
        return template.format(**slots)

    def generate_dataset(self, n=50):
        # 批量生成指令-回复对（无需种子）
        dataset = []
        for _ in range(n):
            info = self.generate_instruction()
            resp = self.generate_response(info)
            dataset.append({
                'instruction': info['instruction'],
                'response': resp,
                'prefix': info['prefix'],
            })
        return dataset

    def _quality_score(self, instr, resp):
        # 质量评分：指令长度 + 词汇丰富度 + 指令-回复相关性
        instr_toks = instr.lower().split()
        resp_toks = resp.lower().split()
        if not instr_toks or not resp_toks:
            return 0.0
        len_score = min(len(instr_toks) / 12, 1.0)
        richness = len(set(instr_toks)) / len(instr_toks)
        overlap = len(set(instr_toks) & set(resp_toks)) / max(len(set(instr_toks)), 1)
        return round(0.4 * len_score + 0.3 * richness + 0.3 * overlap, 3)

    def filter_quality(self, dataset, min_instr_len=5, min_resp_len=8):
        # 过滤低质量样本：长度过短、指令重复
        seen = set()
        kept = []
        for item in dataset:
            instr = item['instruction']
            resp = item['response']
            if len(instr.split()) < min_instr_len:
                continue
            if len(resp.split()) < min_resp_len:
                continue
            if instr in seen:
                continue
            seen.add(instr)
            item['quality'] = self._quality_score(instr, resp)
            kept.append(item)
        return kept


print('=== Magpie 数据生成方法 ===')

magpie = MagpieGenerator()

# 演示单条 Magpie 生成流程
print('\n--- 单条 Magpie 生成流程 ---')
info = magpie.generate_instruction()
print('对话前缀: ' + info['prefix'])
print('完整模板: ' + info['full_template'])
print('生成指令: ' + info['instruction'])
response = magpie.generate_response(info)
print('生成回复: ' + response)

# 批量生成 50 条指令-回复对
print('\n--- 批量生成 50 条指令-回复对 ---')
dataset = magpie.generate_dataset(n=50)
print('生成总数: ' + str(len(dataset)))

# 质量过滤
filtered = magpie.filter_quality(dataset, min_instr_len=5, min_resp_len=8)
print('过滤后: ' + str(len(filtered)))

# 质量分布
scores = [item['quality'] for item in filtered]
quality_bins = Counter()
for s in scores:
    if s >= 0.6:
        quality_bins['high (>=0.6)'] += 1
    elif s >= 0.4:
        quality_bins['medium (0.4-0.6)'] += 1
    else:
        quality_bins['low (<0.4)'] += 1

print('\n质量分布:')
for label in ['high (>=0.6)', 'medium (0.4-0.6)', 'low (<0.4)']:
    count = quality_bins.get(label, 0)
    bar = '#' * count
    print('  ' + label + ': ' + str(count) + ' ' + bar)

print('\n样本展示 (前 5 条):')
for i, item in enumerate(filtered[:5], 1):
    print('  ' + str(i) + '. [' + str(item['quality']) + '] ' + item['instruction'])
    print('     -> ' + item['response'][:70])

# 与 Self-Instruct 对比（需要种子 vs 无需种子）
print('\n--- Magpie vs Self-Instruct 对比 ---')

# Self-Instruct 必须依赖种子指令
seeds = [
    'write a poem about nature',
    'translate this sentence to chinese',
    'summarize the article below',
]
print('Self-Instruct 种子数: ' + str(len(seeds)) + ' (需要种子引导)')
print('Magpie 种子数: 0 (仅需对话前缀, 无需种子)')

# 模拟 Self-Instruct 生成（基于种子变换）
si_generated = []
for seed in seeds:
    for _ in range(8):
        suffix = random.choice([' with examples', ' in brief', ' step by step',
                                ' concisely', ' with details'])
        si_generated.append(seed + suffix)
si_unique = list(dict.fromkeys(si_generated))

# Magpie 生成（不依赖种子）
magpie_generated = [item['instruction'] for item in filtered]


def lexical_diversity(samples):
    toks = []
    for s in samples:
        toks.extend(s.lower().split())
    if not toks:
        return 0.0
    return round(len(set(toks)) / len(toks), 3)


print('\n生成对比:')
print('  Self-Instruct 生成数: ' + str(len(si_generated)))
print('    独立指令数: ' + str(len(si_unique)))
print('    词汇多样性: ' + str(lexical_diversity(si_generated)))
print('  Magpie 生成数: ' + str(len(magpie_generated)))
print('    独立指令数: ' + str(len(set(magpie_generated))))
print('    词汇多样性: ' + str(lexical_diversity(magpie_generated)))

# 方法特性对比表
print('\n方法特性对比:')
rows = [
    ('需要种子', '是', '否'),
    ('依赖对齐特性', '否', '是'),
    ('指令来源', '种子+模板变换', 'LLM自动续写'),
    ('多样性', '受种子限制', '接近真实分布'),
    ('自动化程度', '半自动', '完全自动'),
]
header = '  ' + '特性'.ljust(15) + 'Self-Instruct'.ljust(22) + 'Magpie'.ljust(18)
print(header)
print('  ' + '-' * 55)
for feat, si, mg in rows:
    print('  ' + feat.ljust(15) + si.ljust(22) + mg.ljust(18))

print(f'\nKey: Magpie 利用对齐 LLM 的对话模板前缀自动生成指令，无需种子，多样性更接近真实用户输入。')

## 📝 课后思考题

1. 回译在什么情况下会引入语义漂移？如何检测并缓解？
2. Self-Instruct 生成的指令为什么必须做去重？如果不去重会带来什么训练问题？
3. Evol-Instruct 的四种演化方向（深化、简化、约束、改写）各适合什么场景？如何组合使用？
4. 对话数据增强中，如何保证增强后的多轮对话仍然连贯？设计一个一致性检查方法。
5. 在 RLHF 偏好数据增强中，应优先保证保真度还是多样性？为什么？